In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

xl = pd.ExcelFile('/Users/headofthetable/inventory-optimisation/data/raw/Apparel_Store_Inventory_JIT_Dataset.xlsx')
sales = pd.read_excel(xl, 'Daily_Sales_Transactions', header=3)
products = pd.read_excel(xl, 'Product_Master', header=3)
inventory = pd.read_excel(xl, 'Inventory_Snapshot', header=3)

sales['Date'] = pd.to_datetime(sales['Date'])
sales = sales.sort_values('Date').reset_index(drop=True)

print("Sales:", sales.shape)
print("Products:", products.shape)
print("Inventory:", inventory.shape)
print()
print("Inventory columns:", inventory.columns.tolist())

Sales: (45423, 11)
Products: (576, 19)
Inventory: (576, 7)

Inventory columns: ['SKU', 'Opening Stock (1-Jan-2026)', 'Reorder Point (Units)', 'Reorder Qty (Units)', 'Closing Stock (30-Jun-2026)', 'In-Transit Qty', 'In-Transit Expected Date']


In [2]:
print(inventory.head(10))
print()
print("Closing stock range:", 
      inventory['Closing Stock (30-Jun-2026)'].min(),
      "to",
      inventory['Closing Stock (30-Jun-2026)'].max())
print()
print("SKUs with zero closing stock:", 
      (inventory['Closing Stock (30-Jun-2026)'] == 0).sum())

       SKU  Opening Stock (1-Jan-2026)  Reorder Point (Units)  \
0  AP-1001                          83                     15   
1  AP-1002                          77                     18   
2  AP-1003                          98                     18   
3  AP-1004                         102                     17   
4  AP-1005                          96                     15   
5  AP-1006                         107                     14   
6  AP-1007                         115                     15   
7  AP-1008                          70                     15   
8  AP-1009                         100                     15   
9  AP-1010                          64                     15   

   Reorder Qty (Units)  Closing Stock (30-Jun-2026)  In-Transit Qty  \
0                   60                           32               0   
1                   60                           23               0   
2                   60                           41               0   


In [3]:
# Merge inventory with product info
inv_full = inventory.merge(
    products[['SKU', 'Sub-Category', 'Brand Line', 
              'Lead Time (Days)', 'MOQ (Units)']],
    on='SKU', how='left'
)

# Calculate daily sales velocity per SKU
daily_velocity = (
    sales.groupby('SKU')['Units Sold']
    .agg(['mean', 'std'])
    .reset_index()
)
daily_velocity.columns = ['SKU', 'avg_daily_sales', 'std_daily_sales']
daily_velocity['std_daily_sales'] = daily_velocity['std_daily_sales'].fillna(0)

# Merge into inventory
inv_full = inv_full.merge(daily_velocity, on='SKU', how='left')

print("Combined inventory data:")
print(inv_full[['SKU', 'Closing Stock (30-Jun-2026)', 
                'Lead Time (Days)', 'MOQ (Units)',
                'avg_daily_sales', 'std_daily_sales']].head(10))

Combined inventory data:
       SKU  Closing Stock (30-Jun-2026)  Lead Time (Days)  MOQ (Units)  \
0  AP-1001                           32                18           60   
1  AP-1002                           23                18           60   
2  AP-1003                           41                18           60   
3  AP-1004                           52                18           60   
4  AP-1005                           42                18           60   
5  AP-1006                           54                18           60   
6  AP-1007                            0                18           60   
7  AP-1008                           22                18           60   
8  AP-1009                           50                18           60   
9  AP-1010                           62                18           60   

   avg_daily_sales  std_daily_sales  
0         1.321429         0.541322  
1         1.390244         0.603613  
2         1.271739         0.515757  
3       

In [4]:
# Calculate safety stock, reorder point, order quantity
Z = 1.65  # 95% service level

inv_full['safety_stock'] = (
    Z * inv_full['std_daily_sales'] * 
    np.sqrt(inv_full['Lead Time (Days)'])
).round(0)

inv_full['reorder_point'] = (
    inv_full['avg_daily_sales'] * inv_full['Lead Time (Days)'] + 
    inv_full['safety_stock']
).round(0)

# Forecast next 30 days demand
inv_full['forecast_30_days'] = (
    inv_full['avg_daily_sales'] * 30
).round(0)

# Order quantity
inv_full['order_qty'] = (
    inv_full['forecast_30_days'] + 
    inv_full['safety_stock'] - 
    inv_full['Closing Stock (30-Jun-2026)']
).clip(lower=inv_full['MOQ (Units)'])
inv_full['order_qty'] = inv_full['order_qty'].round(0)

# Show results
print(inv_full[['SKU', 'avg_daily_sales', 'safety_stock',
                'reorder_point', 'forecast_30_days',
                'Closing Stock (30-Jun-2026)',
                'order_qty']].head(10))

       SKU  avg_daily_sales  safety_stock  reorder_point  forecast_30_days  \
0  AP-1001         1.321429           4.0           28.0              40.0   
1  AP-1002         1.390244           4.0           29.0              42.0   
2  AP-1003         1.271739           4.0           27.0              38.0   
3  AP-1004         1.264368           3.0           26.0              38.0   
4  AP-1005         1.280899           4.0           27.0              38.0   
5  AP-1006         1.345238           4.0           28.0              40.0   
6  AP-1007         1.419753           6.0           32.0              43.0   
7  AP-1008         1.241379           4.0           26.0              37.0   
8  AP-1009         1.392405           6.0           31.0              42.0   
9  AP-1010         1.340659           4.0           28.0              40.0   

   Closing Stock (30-Jun-2026)  order_qty  
0                           32       60.0  
1                           23       60.0  
2        

In [5]:
# Add cost price for working capital calculation
inv_full = inv_full.merge(
    products[['SKU', 'Cost Price (INR)', 'Velocity Tier']],
    on='SKU', how='left'
)

# Working capital = order quantity × cost price
inv_full['working_capital'] = (
    inv_full['order_qty'] * inv_full['Cost Price (INR)']
).round(0)

# Flag urgent SKUs — closing stock below reorder point
inv_full['urgent'] = (
    inv_full['Closing Stock (30-Jun-2026)'] <= 
    inv_full['reorder_point']
)

# Summary
print("=== Inventory Recommendation Summary ===")
print(f"Total SKUs: {len(inv_full)}")
print(f"Urgent reorder needed: {inv_full['urgent'].sum()} SKUs")
print(f"Total working capital required: INR {inv_full['working_capital'].sum():,.0f}")
print()
print("Top 10 urgent SKUs:")
urgent_skus = inv_full[inv_full['urgent']].nlargest(10, 'working_capital')
print(urgent_skus[['SKU', 'Closing Stock (30-Jun-2026)',
                    'reorder_point', 'order_qty', 
                    'working_capital', 'Velocity Tier']].to_string())

=== Inventory Recommendation Summary ===
Total SKUs: 576
Urgent reorder needed: 259 SKUs
Total working capital required: INR 23,723,147

Top 10 urgent SKUs:
         SKU  Closing Stock (30-Jun-2026)  reorder_point  order_qty  working_capital Velocity Tier
157  AP-1158                           10           40.0       62.0         141732.0    Fast Mover
159  AP-1160                            7           37.0       61.0         139446.0    Fast Mover
150  AP-1151                           19           39.0       60.0         137160.0    Fast Mover
152  AP-1153                           13           36.0       60.0         137160.0    Fast Mover
154  AP-1155                           22           37.0       60.0         137160.0    Fast Mover
156  AP-1157                           31           36.0       60.0         137160.0    Fast Mover
166  AP-1167                           22           34.0       48.0         105072.0  Medium Mover
168  AP-1169                           28          

## Inventory Recommendation Engine Results
- 259 SKUs need urgent reordering (stock below reorder point)
- Total working capital required: INR 23,723,147
- Top urgent SKUs are Fast Movers — highest velocity products
  running low first as expected
- Safety stock calculated at 95% service level (Z=1.65)
- Order quantities floored at MOQ (60 units minimum)
- This output directly answers the business question:
  what to order, how much, and how urgently